# Assignment: Building a Modular Data Sanitization & Exploration Engine

### Completed Solution Notebook
This notebook provides the fully implemented `PlottingMethods` and `DataInspector` classes alongside a comprehensive real-world demonstration flow using the Titanic dataset.

## 1. Importing Dependencies
First, we import the necessary libraries for data processing, statistical computations, and interactive Plotly visualizations.

In [ ]:
import io
import numpy as np
import pandas as pd
import scipy.stats as stats
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
try:
    from google.colab import files
except ImportError:
    files = None

## 2. Implementing the PlottingMethods Class
This class isolates granular chart generation and returns HTML-wrapped interactive visualizations.

In [ ]:
class PlottingMethods:
    """
    A modular class containing static methods for granular chart generation using Plotly.
    All methods return HTML strings or display standalone interactive figures.
    """
    
    @staticmethod
    def generate_bar_chart(df, column, title="Bar Chart"):
        """
        Generates an interactive bar chart showing frequency and percentage.
        """
        if df is None or column not in df.columns:
            return "<p>Invalid DataFrame or Column</p>"
        
        counts = df[column].value_counts(dropna=True).reset_index()
        counts.columns = [column, 'Count']
        total = counts['Count'].sum()
        counts['Percentage'] = (counts['Count'] / total * 100).round(2)
        counts['Label'] = counts.apply(lambda row: f"{row['Count']} ({row['Percentage']}%%)", axis=1)
        
        fig = px.bar(counts, x=column, y='Count', text='Label', title=title,
                     labels={'Count': 'Frequency', column: str(column).capitalize()},
                     template='plotly_white')
        fig.update_traces(textposition='outside', marker_color='rgb(55, 83, 109)')
        return fig.to_html(include_plotlyjs='cdn')
        
    @staticmethod
    def generate_pie_chart(df, column, title="Pie Chart"):
        """
        Generates an interactive pie chart with percentage labels.
        """
        if df is None or column not in df.columns:
            return "<p>Invalid DataFrame or Column</p>"
            
        counts = df[column].value_counts(dropna=True).reset_index()
        counts.columns = [column, 'Count']
        
        fig = px.pie(counts, names=column, values='Count', title=title, template='plotly_white')
        fig.update_traces(textinfo='percent+label')
        return fig.to_html(include_plotlyjs='cdn')
        
    @staticmethod
    def generate_histogram(df, column, bins=30, title="Histogram"):
        """
        Generates an interactive histogram for numeric distribution.
        """
        if df is None or column not in df.columns:
            return "<p>Invalid DataFrame or Column</p>"
            
        fig = px.histogram(df, x=column, nbins=bins, title=title, template='plotly_white',
                           labels={column: str(column).capitalize()},
                           color_discrete_sequence=['teal'])
        fig.update_layout(yaxis_title="Frequency")
        return fig.to_html(include_plotlyjs='cdn')

## 3. Implementing the Core DataInspector Class
This encapsulates data ingestion, advanced sanitization, intelligent profiling, structural cleaning, scaling, and deep multi-type statistical visualization.

In [ ]:
class DataInspector:
    """
    An end-to-end framework for CSV ingestion, automated sanitization, 
    advanced descriptive profiling, interactive cleaning, engineering scaling, and statistical associations.
    """
    
    def __init__(self):
        self.df = None
        self.garbage_strings = ['?', 'n/a', 'N/A', 'null', 'NULL', ' ', 'None', 'none', '-']
        
    def upload_data(self):
        """
        Handles local file uploads directly in Google Colab environment or fallbacks cleanly.
        """
        if files is not None:
            print("Please upload your CSV target dataset:")
            uploaded = files.upload()
            if uploaded:
                filename = list(uploaded.keys())[0]
                self.df = pd.read_csv(io.BytesIO(uploaded[filename]))
                print(f"Successfully loaded local file: {filename} ({self.df.shape[0]} rows, {self.df.shape[1]} columns).")
                self._sanitize_dataset()
            else:
                print("Upload cancelled.")
        else:
            print("Standard system mode detected (Not inside interactive Google Colab environment).")
            print("Please use load_dataframe(df) or pass a filepath directly via load_csv(path).")
            
    def load_csv(self, filepath):
        """
        Loads data from a local filesystem path or URL.
        """
        self.df = pd.read_csv(filepath)
        print(f"Successfully loaded dataset: {self.df.shape[0]} rows, {self.df.shape[1]} columns.")
        self._sanitize_dataset()
        
    def load_dataframe(self, dataframe):
        """
        Directly attaches an existing active dataframe instance.
        """
        self.df = dataframe.copy()
        print(f"Attached active dataframe instance: {self.df.shape[0]} rows, {self.df.shape[1]} columns.")
        self._sanitize_dataset()

    def _sanitize_dataset(self):
        """
        Internal pipeline method: Automatically cleans garbage text patterns into actual NaNs 
        and intelligently converts potential implicit numerical schemas.
        """
        if self.df is None: return
        
        # Replace predefined garbage string entities
        for s in self.garbage_strings:
            self.df.replace(s, np.nan, inplace=True)
            
        # Strip whitespaces if any remain in object columns
        for col in self.df.select_dtypes(include=['object']):
            self.df[col] = self.df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
            self.df[col].replace('', np.nan, inplace=True)
            
        # Intelligent structural column-type coercion safely
        for col in self.df.columns:
            if self.df[col].dtype == 'object':
                try:
                    coerced = pd.to_numeric(self.df[col], errors='coerce')
                    # Force conversion if it doesn't resolve to a completely empty matrix structure
                    if coerced.notna().sum() > 0:
                        self.df[col] = coerced
                except Exception:
                    pass
        print("Dataset sanitization and auto-type mapping complete.")
        
    def display_summary(self):
        """
        Displays comprehensive metrics, dimension metadata, explicit structural splits and structural data previews.
        """
        if self.df is None:
            print("No active dataset found.")
            return
            
        num_cols = list(self.df.select_dtypes(include=[np.number]).columns)
        cat_cols = list(self.df.select_dtypes(exclude=[np.number]).columns)
        
        print("="*60)
        print("                   DATA INSPECTOR METRICS EXPLICIT")
        print("="*60)
        print(f"Total Row Records     : {self.df.shape[0]}")
        print(f"Total Column Features : {self.df.shape[1]}")
        print(f"Numeric Column Count  : {len(num_cols)} -> {num_cols}")
        print(f"Categorical Count     : {len(cat_cols)} -> {cat_cols}")
        print("-"*60)
        print("\nMissing Value Breakdown per Feature:")
        missing_info = self.df.isnull().sum()
        for k, v in missing_info.items():
            if v > 0:
                pct = (v / self.df.shape[0] * 100).round(2)
                print(f" - {k}: {v} null entities detected ({pct}%%)")
        if missing_info.sum() == 0:
            print(" -> No missing values found across features.")
            
        print("\nFirst 20 Data Rows Preview:")
        display(self.df.head(20))
        
    def handle_missing_values(self, column, strategy='mean', constant_value=None):
        """
        Imputes missing indicators inside requested vectors based on flexible chosen algorithmic schemas.
        """
        if self.df is None or column not in self.df.columns:
            print(f"Error: Feature {column} not located inside data.")
            return
            
        if self.df[column].isnull().sum() == 0:
            print(f"Feature '{column}' exhibits zero null variants. Optimization omitted.")
            return
            
        if strategy == 'mean':
            val = self.df[column].mean()
            self.df[column].fillna(val, inplace=True)
        elif strategy == 'median':
            val = self.df[column].median()
            self.df[column].fillna(val, inplace=True)
        elif strategy == 'mode':
            val = self.df[column].mode()[0]
            self.df[column].fillna(val, inplace=True)
        elif strategy == 'constant':
            self.df[column].fillna(constant_value, inplace=True)
        else:
            print(f"Imputation Strategy schema variant '{strategy}' unrecognized.")
            return
        print(f"Successfully executed target imputation on '{column}' using strategy '{strategy}' (Value: {val if strategy != 'constant' else constant_value}).")

    def remove_duplicates(self):
        """
        Prunes duplicate row configurations.
        """
        if self.df is None: return
        before = self.df.shape[0]
        self.df.drop_duplicates(inplace=True)
        after = self.df.shape[0]
        print(f"Duplicate filter processing finalized: Eliminated {before - after} matching row patterns.")
        
    def handle_outliers(self, column, action='flag', threshold=1.5):
        """
        Detects numeric outliers via IQR bounds strategy. Actions include: 'flag' or 'delete'.
        """
        if self.df is None or column not in self.df.columns:
            print("Target feature array invalid.")
            return
            
        if not np.issubdtype(self.df[column].dtype, np.number):
            print("Outlier analysis restricted strictly to quantitative data frameworks.")
            return
            
        q1 = self.df[column].quantile(0.25)
        q3 = self.df[column].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - threshold * iqr
        upper_bound = q3 + threshold * iqr
        
        outliers_mask = (self.df[column] < lower_bound) | (self.df[column] > upper_bound)
        count = outliers_mask.sum()
        
        print(f"IQR Analysis for '{column}': [{lower_bound:.3f} to {upper_bound:.3f}]. Identified {count} extreme instances.")
        
        if action == 'delete':
            self.df = self.df[~outliers_mask].reset_index(drop=True)
            print(f"Successfully deleted {count} rows flagged as structural outliers from active operational data.")
        elif action == 'flag':
            self.df[f'{column}_outlier'] = outliers_mask.astype(int)
            print(f"Created logical accent mapping flag: '{column}_outlier'.")
            
    def delete_rows(self, row_indices_str):
        """
        Prunes rows dynamically via a comma-separated string formatting configuration.
        """
        if self.df is None: return
        try:
            indices = [int(x.strip()) for x in row_indices_str.split(',') if x.strip().isdigit()]
            valid_indices = [idx for idx in indices if idx in self.df.index]
            self.df.drop(index=valid_indices, inplace=True)
            self.df.reset_index(drop=True, inplace=True)
            print(f"Pruned {len(valid_indices)} manual specific rows from structured frame.")
        except Exception as e:
            print(f"Failed to process structural row elimination metrics: {e}")
            
    def delete_columns(self, column_names_str):
        """
        Prunes feature columns based on comma-separated layout criteria.
        """
        if self.df is None: return
        cols = [x.strip() for x in column_names_str.split(',') if x.strip()]
        valid_cols = [c for c in cols if c in self.df.columns]
        self.df.drop(columns=valid_cols, inplace=True)
        print(f"Pruned feature schemas: {valid_cols}.")
        
    def extract_normalized_numeric_data(self, columns, strategy='standard'):
        """
        Extracts scaled representation of targeted numerical vectors. Strategies: 'minmax', 'standard', 'robust'.
        """
        if self.df is None:
            return pd.DataFrame()
        valid_cols = [c for c in columns if c in self.df.columns and np.issubdtype(self.df[c].dtype, np.number)]
        if not valid_cols:
            return pd.DataFrame()
            
        sub_df = self.df[valid_cols].copy()
        # Handle internal runtime missing variables safely prior to encoding scaling structures
        sub_df = sub_df.fillna(sub_df.median())
        
        if strategy == 'minmax':
            scaler = MinMaxScaler()
        elif strategy == 'robust':
            scaler = RobustScaler()
        else:
            scaler = StandardScaler()
            
        scaled_matrix = scaler.fit_transform(sub_df)
        scaled_df = pd.DataFrame(scaled_matrix, columns=[f"{c}_scaled" for c in valid_cols], index=self.df.index)
        return scaled_df
        
    def extract_normalized_categorical_data(self, columns, strategy='onehot'):
        """
        Extracts numeric-mapped formats of categories. Strategies: 'onehot', 'ordinal', 'uniform'.
        """
        if self.df is None:
            return pd.DataFrame()
        valid_cols = [c for c in columns if c in self.df.columns]
        if not valid_cols:
            return pd.DataFrame()
            
        sub_df = self.df[valid_cols].copy().astype(str)
        
        if strategy == 'onehot':
            encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
            encoded_matrix = encoder.fit_transform(sub_df)
            feature_names = encoder.get_feature_names_out(valid_cols)
            return pd.DataFrame(encoded_matrix, columns=feature_names, index=self.df.index)
            
        elif strategy == 'ordinal':
            encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            encoded_matrix = encoder.fit_transform(sub_df)
            return pd.DataFrame(encoded_matrix, columns=[f"{c}_encoded" for c in valid_cols], index=self.df.index)
            
        elif strategy == 'uniform':
            # Scaled Ordinal formatting mapping precisely inside [0, 1] range constraints
            encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            encoded_matrix = encoder.fit_transform(sub_df)
            uniform_matrix = np.zeros_like(encoded_matrix, dtype=float)
            for i in range(encoded_matrix.shape[1]):
                col_vec = encoded_matrix[:, i]
                max_val = np.max(col_vec)
                if max_val > 0:
                    uniform_matrix[:, i] = col_vec / max_val
                else:
                    uniform_matrix[:, i] = col_vec
            return pd.DataFrame(uniform_matrix, columns=[f"{c}_uniform" for c in valid_cols], index=self.df.index)
        
        return pd.DataFrame()
        
    def merge_engineered_datasets(self, numeric_cols, numeric_strategy, categorical_cols, categorical_strategy):
        """
        Assembles a integrated unified pipeline dataframe containing raw inputs alongside engineered scales.
        """
        num_df = self.extract_normalized_numeric_data(numeric_cols, numeric_strategy)
        cat_df = self.extract_normalized_categorical_data(categorical_cols, categorical_strategy)
        
        combined = self.df.copy()
        if not num_df.empty:
            combined = combined.join(num_df)
        if not cat_df.empty:
            combined = combined.join(cat_df)
        return combined

    def plot_univariate_subplots(self, column):
        """
        Generates an extensive 3-panel statistical subplot dashboard for an engineering feature column.
        """
        if self.df is None or column not in self.df.columns:
            print("Column not found.")
            return
        if not np.issubdtype(self.df[column].dtype, np.number):
            print("Univariate subplots require standard numerical metrics layout.")
            return
            
        cleaned_data = self.df[column].dropna()
        
        fig = make_subplots(
            rows=3, cols=1,
            subplot_titles=("Horizontal Box & Violin Distribution Profile", 
                            "Sequential Observation Index vs Feature Value Plot", 
                            "Probability Frequency Density Histogram"),
            vertical_spacing=0.12
        )
        
        # Panel 1: Violin + Box plot overlay
        fig.add_trace(go.Violin(x=cleaned_data, name="Violin", box_visible=True, meanline_visible=True, marker_color='indigo'), row=1, col=1)
        
        # Panel 2: Scatter plot (Index vs Value)
        fig.add_trace(go.Scatter(y=cleaned_data, mode='markers', marker=dict(color='deepskyblue', opacity=0.7), name="Scatter"), row=2, col=1)
        
        # Panel 3: Histogram
        fig.add_trace(go.Histogram(x=cleaned_data, nbinsx=30, marker_color='crimson', name="Histogram"), row=3, col=1)
        
        fig.update_layout(height=850, width=900, title_text=f"Advanced Univariate Analytical Engine Dashboard: '{column}'", template="plotly_white", showlegend=False)
        fig.show()
        
    def plot_relationship(self, x_col, y_col):
        """
        Intelligently detects feature schema layouts and determines the optimal analytical chart representation.
        """
        if self.df is None or x_col not in self.df.columns or y_col not in self.df.columns:
            print("Input feature coordinate schemas invalid.")
            return
            
        x_is_num = np.issubdtype(self.df[x_col].dtype, np.number)
        y_is_num = np.issubdtype(self.df[y_col].dtype, np.number)
        
        # Case 1: Numeric-Numeric Relationship
        if x_is_num and y_is_num:
            print(f"Mapping Analytical Schema: Numeric-to-Numeric Detected for ({x_col} vs {y_col}). Producing Scatter Plot with OLS Fit line.")
            # Drop runtime null representations to permit robust OLS regression fit line computing
            temp = self.df[[x_col, y_col]].dropna()
            fig = px.scatter(temp, x=x_col, y=y_col, trendline="ols", title=f"Bivariate Numerical Analysis: {x_col} vs {y_col}", template='plotly_white', color_discrete_sequence=['darkblue'])
            fig.show()
            
        # Case 2: Mixed Data Relationship (Categorical vs Numeric)
        elif (x_is_num and not y_is_num) or (not x_is_num and y_is_num):
            cat = x_col if not x_is_num else y_col
            num = y_col if x_is_num else x_col
            print(f"Mapping Analytical Schema: Categorical/Numerical Contrast Detected ({cat} vs {num}). Generating Structural Whisker Box Plot.")
            fig = px.box(self.df, x=cat, y=num, points="all", title=f"Mixed Structural Distribution: {cat} vs {num}", template='plotly_white', color=cat)
            fig.show()
            
        # Case 3: Categorical-Categorical Relationship
        else:
            print(f"Mapping Analytical Schema: Categorical-to-Categorical Cross Tabulation Detected for ({x_col} vs {y_col}). Rendering Multi-Grouped Bar Configuration.")
            counts = self.df.groupby([x_col, y_col]).size().reset_index(name='Frequency')
            fig = px.bar(counts, x=x_col, y='Frequency', color=y_col, barmode='group', title=f"Cross-Categorical Analysis: {x_col} vs {y_col}", template='plotly_white')
            fig.show()
            
    def plot_all_associations_heatmap(self):
        """
        Advanced statistical computation suite: Dynamically calculates explicit continuous and nominal associations 
        (Pearson's r, Cramer's V, and Correlation Ratio Eta via ANOVA) across all variables to form a unified correlation matrix.
        """
        if self.df is None:
            return
            
        # Drop flags created on the fly during outlier detection pipelines to keep focus clear
        cols = [c for c in self.df.columns if not str(c).endswith('_outlier')]
        n = len(cols)
        
        # Initialize an empty square association matrix tracking dataframe architecture
        assoc_matrix = pd.DataFrame(np.zeros((n, n)), index=cols, columns=cols)
        
        for i in range(n):
            for j in range(n):
                col1 = cols[i]
                col2 = cols[j]
                
                if i == j:
                    assoc_matrix.iloc[i, j] = 1.0
                    continue
                    
                is_num1 = np.issubdtype(self.df[col1].dtype, np.number)
                is_num2 = np.issubdtype(self.df[col2].dtype, np.number)
                
                # Extract paired observations while cleaning structural missing values
                temp = self.df[[col1, col2]].dropna()
                if temp.empty:
                    assoc_matrix.iloc[i, j] = 0.0
                    continue
                    
                # Metric 1: Numeric vs Numeric -> Pearson's Correlation Coefficient
                if is_num1 and is_num2:
                    if temp[col1].std() == 0 or temp[col2].std() == 0:
                        r = 0.0
                    else:
                        r, _ = stats.pearsonr(temp[col1], temp[col2])
                    assoc_matrix.iloc[i, j] = round(r, 3)
                    
                # Metric 2: Categorical vs Categorical -> Cramér's V Coefficient
                elif not is_num1 and not is_num2:
                    confusion_matrix = pd.crosstab(temp[col1], temp[col2])
                    if confusion_matrix.size == 0 or confusion_matrix.sum().sum() == 0:
                        v = 0.0
                    else:
                        chi2 = stats.chi2_contingency(confusion_matrix)[0]
                        n_obs = confusion_matrix.sum().sum()
                        r_dims, c_dims = confusion_matrix.shape
                        min_dim = min(r_dims - 1, c_dims - 1)
                        if min_dim == 0:
                            v = 0.0
                        else:
                            v = np.sqrt(chi2 / (n_obs * min_dim))
                    assoc_matrix.iloc[i, j] = round(v, 3)
                    
                # Metric 3: Mixed (Numeric & Categorical) -> Correlation Ratio (Eta)
                else:
                    num_col = col1 if is_num1 else col2
                    cat_col = col2 if is_num1 else col1
                    
                    groups = [group[num_col].values for name, group in temp.groupby(cat_col)]
                    # Filter out empty dimensional groups to prevent evaluation crashes
                    groups = [g for g in groups if len(g) > 0]
                    
                    if len(groups) <= 1:
                        eta = 0.0
                    else:
                        all_vals = np.concatenate(groups)
                        grand_mean = np.mean(all_vals)
                        ss_total = np.sum((all_vals - grand_mean) ** 2)
                        
                        ss_between = 0.0
                        for g in groups:
                            ss_between += len(g) * (np.mean(g) - grand_mean) ** 2
                            
                        if ss_total == 0:
                            eta = 0.0
                        else:
                            eta = np.sqrt(ss_between / ss_total)
                    assoc_matrix.iloc[i, j] = round(eta, 3)
                    
        fig = px.imshow(
            assoc_matrix, 
            text_auto=True, 
            aspect="auto", 
            color_continuous_scale='RdBu_r',
            zmin=-1.0, zmax=1.0,
            title="Unified Multimodal Statistical Association Map (Pearson r / Cramér V / Eta)",
            template="plotly_white"
        )
        fig.update_layout(height=650, width=750)
        fig.show()

## 4. Operational Pipeline Test & Demonstration
We simulate a real-world analytics workflow using the Titanic dataset to validate the modular processing engine.

In [ ]:
print("Initializing Demo pipeline with Titanic Dataset...")
titanic_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

# Initialize engine
inspector = DataInspector()
inspector.load_csv(titanic_url)

# 1. Display Initial Structured Metrics Profiling
inspector.display_summary()

In [ ]:
# 2. Clean and Impute Missing Values
print("Handling internal structural null vectors via automated strategies...")
inspector.handle_missing_values('Age', strategy='median')
inspector.handle_missing_values('Embarked', strategy='mode')

# 3. Manage and Flag Outliers
inspector.handle_outliers('Fare', action='flag', threshold=1.5)

# 4. Display Post-Cleaning structural overview
inspector.display_summary()

In [ ]:
# 5. Feature Engineering, Normalization & Feature Merging
print("Extracting normalized numerical features and multi-scale categorical encoders...")
scaled_data = inspector.merge_engineered_datasets(
    numeric_cols=['Age', 'Fare'], 
    numeric_strategy='standard', 
    categorical_cols=['Pclass', 'Sex', 'Embarked'], 
    categorical_strategy='onehot'
)
print("\nEngineered Integrated Output Dataframe Sample View:")
display(scaled_data.head(5))

In [ ]:
# 6. Advanced Subplot & Bivariate Profiling
print("Generating univariate subplots for quantitative metrics...")
inspector.plot_univariate_subplots('Age')

print("Generating bivariate smart relationship visualization templates...")
inspector.plot_relationship('Sex', 'Survived') # Categorical vs Numeric

In [ ]:
# 7. Deep Multimodal Heatmap Generation
print("Computing global cross-feature associations matrix heatmap...")
inspector.plot_all_associations_heatmap()

In [ ]:
# 8. Verify HTML Plotting Methods output integration
print("Testing modular html output wrappers via PlottingMethods...")
html_bar = PlottingMethods.generate_bar_chart(inspector.df, 'Pclass', "Passenger Class Count Summary")
print(f"HTML script block wrapper compiled successfully. Size: {len(html_bar)} characters.")